<a href="https://colab.research.google.com/github/mariaantipina/PythonPractice/blob/main/Antipina_Pandas_Practica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 📚 Вариант 13. Фитнес-клуб (Pandas). Полный курс, 60 задач

---

## 🟢 Уровень 1. Новичок (задачи 1–12)

**Задача 1 (создание Series, dtype, shape/size/ndim).** Фитнес-клуб ведёт учёт стоимости 7 абонементов: `[35000, 25000, 18000, 12000, 45000, 22000, 30000]` рублей. Создайте `pd.Series` из этих значений. Выведите `dtype`, `itemsize`, `shape`, `nbytes`. Дополнительно создайте второй Series из тех же значений с типом `float32` через `astype("float32")` и сравните `dtype`, `itemsize`, `nbytes` с первым. Ожидаемый вывод: для первого — `dtype=int64`, `itemsize=8`, `shape=(7,)`, `nbytes=56`; для второго — `dtype=float32`, `itemsize=4`, `nbytes=28`. В комментариях объясните: почему Pandas выбрал `int64`, а не `int32`; почему `shape` у одномерного Series — кортеж `(7,)`, а не `(7)`; что вернёт `prices.iloc[7]` и почему; как связаны `itemsize`, `nbytes` и `size`; во сколько раз `float32` экономнее `float64` и в каких случаях эта экономия критична (например, при хранении 10 млн записей о продажах абонементов за год); почему `astype("float32")` создаёт новый объект, а не меняет исходный Series на месте.



In [ ]:
prices = pd.Series([35000, 25000, 18000, 12000, 45000, 22000, 30000])
prices_f32 = prices.astype("float32")

print(prices.dtype, prices.dtype.itemsize, prices.shape, prices.nbytes)
print(prices_f32.dtype, prices_f32.dtype.itemsize, prices_f32.shape, prices_f32.nbytes)
print()

# int64: тип по умолчанию для целых на 64-битной платформе, вмещает все значения
# (7,) — кортеж из одного элемента; (7) — просто число в скобках, не кортеж
# prices.iloc[7] -> IndexError: позиции 0..6, выход за границу
# nbytes = itemsize * size: int64 8*7=56, float32 4*7=28
# float32 вдвое экономнее float64 -> важно при миллионах записей
# astype("float32") возвращает НОВЫЙ Series, оригинал не меняется
print()


**Задача 2 (создание DataFrame, атрибуты, просмотр).** Фитнес-клуб хранит 20 клиентов: `client_id`, `name` (Иванов, Петрова, Сидоров, Кузнецова, Смирнов), `membership_type` (годовой/полугодовой/месячный/разовый), `age`, `visits_per_month`. Создайте DataFrame из словаря с сохранением типов. Выведите `shape`, `columns`, `index`, `dtypes`. Выполните просмотр: `head(3)`, `tail(2)`, `sample(3, random_state=42)`, `describe()`, `describe(include="all")`. Ожидаемый вывод: `shape=(20, 5)`; `dtypes` показывает `name=str`, `membership_type=str`, `age=int64`, `visits_per_month=int64`; `describe()` возвращает статистику только для `age` и `visits_per_month`; `describe(include="all")` — для всех столбцов. В комментариях объясните: чем `describe()` для числовых столбцов отличается от `describe(include="all")` (какие дополнительные статистики появляются для строк: `count`, `unique`, `top`, `freq`); что показывает `info()` помимо типов (число не-пустых значений, объём памяти); чем `sample()` предпочтительнее `head()` при первичном осмотре (не смещён к началу файла, если данные отсортированы по типу абонемента); почему `shape` — кортеж, а `size` — число; что произойдёт при `pd.DataFrame({...})`, если все списки разной длины (`ValueError`); как `dtypes` помогает обнаружить проблему с типами после чтения CSV (например, `age` стал `object` из-за пустых значений).


In [ ]:
df = pd.DataFrame({
    "client_id": range(1, 21),
    "name": ["Иванов", "Петрова", "Сидоров", "Кузнецова", "Смирнов"] * 4,
    "membership_type": ["годовой", "полугодовой", "месячный", "разовый"] * 5,
    "age": [25 + i for i in range(20)],
    "visits_per_month": [8 + i % 12 for i in range(20)],
})
print(df.shape, list(df.columns), df.index[:3], df.dtypes.to_dict())
print(df.head(3))
print(df.tail(2))
print(df.sample(3, random_state=42))
print(df.describe())
print(df.describe(include="all"))
print()

# describe() — только числовые столбцы; include="all" добавляет для строк:
#   count, unique, top, freq
# info() — типы, число не-пустых значений, объём памяти
# sample() не смещён к началу (в отличие от head при сортировке по типу)
# shape — кортеж (rows, cols); size — rows*cols (число)
# разные длины списков -> ValueError: arrays must be same length
# dtypes выявит проблему: age стал object (пустые/пробелы в CSV)
print()


**Задача 3 (подбор dtype, nullable-типы, переполнение).** Фитнес-клуб хранит три вида данных: возраст клиента (14–80), число тренировок в месяц (0–40), индекс массы тела (15.0–45.0). Создайте DataFrame из 100 строк. Понизьте типы: `age` → `uint8`, `workouts_per_month` → `uint8`, `bmi` → `float32` через `astype()`. Проверьте переполнение: `np.uint8(200) + np.uint8(100)`. Создайте `pd.Series([100, 200, None], dtype="Int64")` и сравните с `dtype="int64"`. Ожидаемый вывод: `np.uint8(200) + np.uint8(100) = 44`; `Int64`-Series сохраняет `None` как `pd.NA`, а `int64`-Series не может его создать; `memory_usage(deep=True)` после понижения типов сокращается примерно на 50%. В комментариях объясните: чем `Int64` отличается от `int64`; зачем нужен `pd.NA`; как переполнение `uint8` проявляется без ошибки (тихий wrap-around); почему для возраста клиента и числа тренировок `uint8` подходит (максимум 255); в каких случаях `float32` недостаточно точен (BMI — да, но при усреднении тысяч измерений может накапливаться погрешность); как `astype("category")` дополнительно сокращает память для низкокардинальных строк (типы абонементов, залы, статусы).


In [ ]:
df = pd.DataFrame({
    "age": np.random.default_rng(0).integers(14, 81, 100),
    "workouts_per_month": np.random.default_rng(1).integers(0, 41, 100),
    "bmi": np.random.default_rng(2).uniform(15, 45, 100),
})
df["age"] = df["age"].astype("uint8")
df["workouts_per_month"] = df["workouts_per_month"].astype("uint8")
df["bmi"] = df["bmi"].astype("float32")
print(df.dtypes.to_dict())
print(np.uint8(200) + np.uint8(100))          # 44
s = pd.Series([100, 200, None], dtype="Int64")
print(s.tolist(), s.dtype)
print(round(df.memory_usage(deep=True).sum() / 1024, 2), "КБ")
print()

# Int64 (nullable) хранит pd.NA; int64 — только числа (None -> float64)
# pd.NA — единый "пропуск" для nullable типов
# переполнение uint8 — тихий wrap-around по модулю 256: 300 -> 44
# uint8 max=255: возраст<=80, тренировок<=40 — влезает
# float32 ~7 значащих цифр: для BMI хватает, при усреднении тысяч копится ошибка
# category для membership_type/hall/status экономит память при малой кардинальности
print()


**Задача 4 (арифметика, выравнивание, assign, pd.col()).** Создайте два Series выручки по залам с индексами `["G1", "G2", "G3", "G4"]` и `["G2", "G3", "G4", "G5"]`. Сложите их — зафиксируйте `NaN` на несовпадающих метках. Повторите через `.add(other, fill_value=0)`. Создайте `df = pd.DataFrame({"price": [35000, 25000, 18000], "members": [40, 60, 50]})`. Через `assign(revenue=lambda x: x["price"] * x["members"])` добавьте `revenue`. Через `assign(revenue=pd.col("price") * pd.col("members"))` — то же. Сравните результаты. Ожидаемый вывод: `s1 + s2` даёт `G1=NaN`, `G2=…`, `G5=NaN`; `.add(..., fill_value=0)` даёт числа везде; оба `assign` возвращают идентичные DataFrame с новым столбцом `revenue`. В комментариях объясните: почему выравнивание по меткам — ключевая особенность Pandas (данные из разных филиалов клуба с разным набором залов); почему `pd.col()` помечен как «начальная поддержка»; почему `assign` не модифицирует исходный DataFrame (возвращает копию); как `fill_value` влияет на тип результата (может привести к `float`, если появляются `NaN`); в каких случаях `lambda` всё ещё нужен вместо `pd.col()` (сложные выражения, вызовы функций); что произойдёт при сложении Series с полностью непересекающимися индексами (все `NaN`).


In [ ]:
s1 = pd.Series([100000, 80000, 50000, 70000], index=["G1", "G2", "G3", "G4"])
s2 = pd.Series([90000, 60000, 40000, 120000], index=["G2", "G3", "G4", "G5"])
print(s1 + s2)
print(s1.add(s2, fill_value=0))

df = pd.DataFrame({"price": [35000, 25000, 18000], "members": [40, 60, 50]})
print(df.assign(revenue=lambda x: x["price"] * x["members"]))
print(df.assign(revenue=df["price"] * df["members"]))
print()

# выравнивание по меткам: G1/G5 -> NaN (нет в одном из Series)
# pd.col() есть только в Pandas 3.0+; в 2.x используем lambda или df["col"]
# assign не меняет df — возвращает копию
# fill_value=0 сохраняет целый dtype (без NaN)
# lambda нужен для сложных выражений (np.where, вызовы функций)
# непересекающиеся индексы: все значения -> NaN
print()


**Задача 5 (arange, linspace, date_range, CSV).** Сгенерируйте DataFrame из 50 посещений: `visit_id`, `visit_date` (2000–2025), `price`. Создайте `pd.Series(np.arange(2000, 2026))` и `pd.Series(np.linspace(500, 5000, 10))` (стоимость разовых посещений). Сохраните DataFrame в `visits.csv` через `to_csv(index=False)` и прочитайте обратно с `parse_dates=["visit_date"]`, `usecols=["visit_id", "price"]`, `nrows=10`. Проверьте `len(np.arange(0, 1, 0.1))` и сравните с `len(np.linspace(0, 1, 10))`. Ожидаемый вывод: `len(np.arange(0, 1, 0.1))` может вернуть 10 или 9 (зависит от платформы, из-за погрешности float); `len(np.linspace(0, 1, 10)) = 10` всегда. В комментариях объясните: почему `linspace` надёжнее при фиксированной длине; как `index=False` влияет на структуру CSV (не записывается лишний столбец с индексом); зачем `parse_dates` при чтении (превращает строку в `datetime64`); почему `freq="YE"` вместо `freq="Y"` в Pandas 3.0; как `usecols` экономит память (читает только нужные столбцы); что произойдёт при чтении CSV без `dtype` для числовых столбцов (Pandas угадает `int64`/`float64`, но может ошибиться, если есть пропуски).


In [ ]:
df = pd.DataFrame({
    "visit_id": range(1, 51),
    "visit_date": pd.date_range("2024-01-01", periods=50, freq="D").astype(str),
    "price": np.random.default_rng(0).integers(500, 5000, 50),
})
df.to_csv("visits.csv", index=False)
df_read = pd.read_csv("visits.csv",
                      parse_dates=["visit_date"],
                      usecols=["visit_id", "visit_date", "price"],
                      nrows=10)
print(df_read.dtypes.to_dict(), len(df_read))
print(len(np.arange(0, 1, 0.1)), len(np.linspace(0, 1, 10)))
print()

# linspace гарантирует num; arange с float может дать 9/10/11
# index=False — не пишет служебный RangeIndex
# parse_dates превращает строку в datetime64[us]
# ВАЖНО: столбец из parse_dates должен быть в usecols, иначе ValueError
# freq="YE" вместо "Y" в Pandas 3.0 (унификация с "ME"/"QE")
# usecols — читает только нужные столбцы, экономит память
# без dtype: Pandas угадает int64/float64, но с пропусками сделает float64
print()


**Задача 6 (DataFrame, доступ loc/iloc/at/iat).** Создайте `attendance` 3×4 со случайными значениями посещаемости, индекс `["пн", "вт", "ср"]`, столбцы `["йога", "кроссфит", "бассейн", "тренажёрный"]`. Выполните выборку: `attendance.loc["вт"]`, `attendance.iloc[1]`, `attendance.loc["ср", "тренажёрный"]`, `attendance.iloc[2, 2]`, `attendance.at["пн", "йога"]`, `attendance.iat[0, 0]`. Выведите `type(...)` и `shape` каждой. Попробуйте `attendance.loc["чт"]` и `attendance.iloc[5]`. Ожидаемый вывод: `type(attendance.loc["вт"])` — `Series` с `shape=(4,)`; `type(attendance.loc["ср", "тренажёрный"])` — скаляр; `attendance.loc["чт"]` → `KeyError`; `attendance.iloc[5]` → `IndexError`. В комментариях объясните: почему `.at` быстрее `.loc` для одного значения (не создаёт промежуточные объекты); в каких случаях `[]` неоднозначен при числовом индексе (если индекс — целые числа, `df[1]` может быть и меткой, и позицией); почему `attendance[["йога"]]` возвращает DataFrame, а не Series (двойные скобки — список столбцов); как `.loc` обрабатывает срез по меткам в отличие от `.iloc` (включает конечную метку); что вернёт `attendance.loc["пн":"ср"]` и почему конечная метка включается (семантика закрытого интервала у `.loc`).

**Задача 7 (память: memory_usage, Parquet, категории).** Создайте DataFrame из 1 000 000 строк: `visit_id` (int64), `bmi` (float64), `workouts_per_month` (int64). Выведите `memory_usage(deep=True)` в МБ. Понизьте типы через `pd.to_numeric(..., downcast=...)`: `visit_id` → `int32`, `bmi` → `float32`, `workouts_per_month` → `uint8`. Сравните память. Добавьте `hall` (йога/кроссфит/бассейн/тренажёрный) из 4 значений — сравните `memory_usage(deep=True)` для `str` и `category`. Сохраните оба варианта в CSV и Parquet. Ожидаемый вывод: исходно ≈15.26 МБ, после понижения ≈3.81 МБ; `category` для `hall` экономит ≈96% памяти относительно `str`; Parquet в 2–5 раз меньше CSV и в 3–10 раз быстрее. В комментариях объясните: во сколько раз `uint8` экономнее `float64` (в 8 раз); в каких случаях `category` хуже `str` (высокая кардинальность — много уникальных ФИО клиентов); почему `memory_usage(deep=True)` строкового столбца превышает `nbytes` (учитывает объекты Python-строк); как `downcast` выбирает тип (минимальный, вмещающий все значения); почему Parquet сохраняет `category` и `datetime`, а CSV — нет (Parquet хранит метаданные и типы).

**Задача 8 (воспроизводимость, sample).** Создайте `rng = np.random.default_rng(seed=42)` и DataFrame из 100 строк: `client_id`, `renew_prob` (uniform 0–1), `spend_amount` (normal 3000±800). Создайте второй генератор с `seed=42` — сравните. Третий без seed — сравните. Выполните `df.sample(n=5, random_state=42)` дважды. Ожидаемый вывод: два генератора с `seed=42` дают идентичные последовательности; третий без seed — другие; `df.sample(n=5, random_state=42)` возвращает одинаковые строки при повторных вызовах. В комментариях объясните: зачем нужен `seed` (воспроизводимость A/B-тестов по продлению абонементов, отладка); чем `default_rng` лучше `np.random.seed` (не глобальное состояние, потокобезопаснее); как `SeedSequence` помогает для параллельных вычислений (разные потоки — разные seed из одного родителя); чем `df.sample` отличается от `rng.choice` по работе с индексом (`sample` сохраняет исходный индекс, `choice` возвращает позиции); почему воспроизводимость критична в production (одинаковые тесты, аудит продаж абонементов); что произойдёт при `np.random.seed` в многопоточном коде (состояние гонки, непредсказуемые результаты).

**Задача 9 (чтение CSV в деталях).** Создайте CSV посещений с разделителем `;`, кодировкой `utf-8`, пропусками `["", "NA", "-"]`. Прочитайте через `read_csv` с параметрами: `sep=";"`, `encoding="utf-8"`, `dtype={"visit_id": "int32"}`, `parse_dates=["visit_date"]`, `usecols=["visit_id", "visit_date", "price"]`, `nrows=10`, `na_values=["", "NA", "-"]`. Создайте второй CSV с десятичной запятой — прочитайте с `decimal=","`. Прочитайте первый без `dtype` — сравните типы. Ожидаемый вывод: с `dtype` столбец `visit_id` — `int32`, без — `int64`; `parse_dates` даёт `datetime64[us]`, без — `str`; `decimal=","` корректно преобразует `"123,45"` в `123.45`. В комментариях объясните: почему `dtype` при чтении эффективнее `astype` после (не создаётся промежуточный `object`-столбец, меньше памяти); когда `nrows` полезен для тестирования (быстрая проверка структуры); как `skiprows` пропускает метаданные (например, шапку выгрузки из CRM клуба); что произойдёт при неверной кодировке (`UnicodeDecodeError` или «кракозябры»); почему `na_values` нужно указывать явно (иначе `"NA"` может остаться строкой); как `decimal=","` влияет на тип результирующего столбца (`float64`).

**Задача 10 (Excel и JSON).** Сохраните DataFrame в Excel с двумя листами (`Clients`, `Memberships`) через `ExcelWriter`. Прочитайте через `sheet_name=None` — получите словарь; проверьте `type()` и `keys()`. Прочитайте только `Clients`. Сохраните в JSON через `to_json(orient="records", lines=True)` и прочитайте через `read_json(lines=True)`. Создайте вложенный JSON (с `address` и `visits`) и нормализуйте через `pd.json_normalize()`. Ожидаемый вывод: `sheet_name=None` возвращает `dict` с ключами `["Clients", "Memberships"]`; `json_normalize` с `record_path="visits"` разворачивает вложенный список в строки; `max_level=1` останавливает нормализацию на первом уровне. В комментариях объясните: в каких случаях `json_normalize` необходим (ответы API клуба с вложенностью по клиенту и его посещениям); как `record_path` разворачивает вложенный список (каждый элемент — строка); почему `orient="records"` удобнее для API (список объектов); чем `lines=True` отличается от обычного JSON (NDJSON — по одному объекту на строку); как `max_level` ограничивает глубину нормализации; какие типы Excel сохраняет, а какие теряет (теряет `category`, `datetime` с таймзоной, длинные ID > 15 цифр).

**Задача 11 (Parquet, сжатие, фильтры).** Сохраните DataFrame посещений в Parquet. Сравните размер файла и скорость чтения с CSV через `time.perf_counter()` (прогрев + 5 замеров). Прочитайте Parquet с `columns=["visit_id", "visit_date"]` — сравните с полным чтением. Прочитайте с `filters=[("visit_date", ">=", "2025-01-01")]` — проверьте `len(result)`. Сохраните в три сжатых CSV: `compression="gzip"`, `"bz2"`, `"xz"`. Ожидаемый вывод: Parquet в 2–5 раз меньше CSV и в 3–10 раз быстрее; `columns=[...]` ускоряет чтение в 2–3 раза; `filters` возвращает только подходящие строки; `xz` даёт наименьший размер, `gzip` — баланс. В комментариях объясните: почему Parquet быстрее и компактнее (колоночное хранение, сжатие по столбцам); как фильтры работают без чтения всего файла (predicate pushdown — статистика min/max в row groups); какой компрессор даёт лучший размер (`xz`, но медленный); почему Parquet сохраняет `category` и `datetime`; как `columns` влияет на объём (читаются только нужные колонки); в каких случаях `filters` не сработает (фильтр по несортированному столбцу без статистики, сложные выражения).

**Задача 12 (матрицы eye/diag, pd.col()).** Создайте список направлений `["йога", "кроссфит", "бассейн", "тренажёрный"]`. Создайте `pd.DataFrame(np.eye(4), index=categories, columns=categories)`. Создайте диагональную матрицу через `np.diag([1.0, 0.8, 0.6, 0.9])` (например, коэффициенты популярности направлений) и смещённую через `np.diag([0.5, 0.3], k=1)`. Через `assign(trace=pd.col("йога") + pd.col("кроссфит"))` добавьте столбец. Ожидаемый вывод: `np.eye(4)` — диагональ из единиц, остальное нули; `np.diag([1.0, 0.8, 0.6, 0.9])` — эти значения на диагонали; `np.diag([0.5, 0.3], k=1)` — на диагонали выше главной. В комментариях объясните: что делает параметр `k` (смещение диагонали); в каких случаях диагональная матрица эффективнее плотной (только диагональ ненулевая — экономия памяти); почему `np.eye` возвращает `float`, а не `int` (по умолчанию `dtype=float`); как `pd.col()` помогает при вычислении `trace`; чем `np.diag`-конструктор отличается от `np.diag`-извлекателя (передаём 1D → матрица, 2D → диагональ); как метки `index`/`columns` влияют на выравнивание при арифметике.

---



## 🔵 Уровень 2. Ученик (задачи 13–22)

**Задача 13 (index: set_index, reset_index, MultiIndex).** Создайте DataFrame из 20 клиентов с `client_id`, `membership_type`, `price`. Установите `client_id` как индекс через `set_index`. Верните обратно через `reset_index()` и `reset_index(drop=True)`. Создайте MultiIndex через `set_index(["membership_type", "client_id"])`. Выполните выборку: `df.loc["годовой"]`, `df.xs("годовой", level="membership_type")`. Создайте DataFrame с дубликатами индекса `["a", "b", "a", "c"]` — проверьте `index.is_unique`, `index.duplicated()`, `df.loc["a"]`. Ожидаемый вывод: `reset_index()` создаёт столбец `client_id` и `RangeIndex`, `reset_index(drop=True)` — только `RangeIndex`; `df.loc["a"]` с дубликатами возвращает две строки; `index.is_unique = False`, `duplicated() = [False, False, True, False]`. В комментариях объясните: чем `drop=True/False` различаются; как `xs` с `drop_level=False` сохраняет уровень; почему `.loc` с дубликатами возвращает несколько строк; как MultiIndex влияет на `groupby` (группировка по типу абонемента и клиенту); что произойдёт при `xs` с несуществующим уровнем (`KeyError`); зачем нужен `allows_duplicate_labels=False` (защита от случайных дубликатов ID клиентов).


In [ ]:
df = pd.DataFrame({
    "client_id": range(1, 21),
    "membership_type": ["годовой", "полугодовой", "месячный", "разовый"] * 5,
    "price": [35000 + i * 500 for i in range(20)],
})
df = df.set_index("client_id")
print(df.index[:3])
print(df.reset_index().columns.tolist())
print(df.reset_index(drop=True).index[:3])
df_mi = df.reset_index().set_index(["membership_type", "client_id"])
print(df_mi.loc["годовой"].head(2))
print(df_mi.xs("годовой", level="membership_type").shape)
print()

# drop=True — старый индекс выброшен; drop=False — становится столбцом
# xs(drop_level=False) сохраняет уровень в результате
# .loc с дубликатами -> DataFrame из нескольких строк
# MultiIndex влияет на groupby: группировка по нескольким ключам сразу
# xs с несуществующим уровнем -> KeyError
# allows_duplicate_labels=False защищает от случайных дубликатов индекса
print()


**Задача 14 (срезы, Copy-on-Write, chained assignment).** Создайте Series из 30 цен абонементов. Выполните срезы: `iloc[:7]`, `iloc[-7:]`, `iloc[10:20]`, `iloc[::2]`, `iloc[::-1]`. Создайте `subset = prices.iloc[2:5]`, измените `subset.iloc[0] = 999` — проверьте исходный Series. Попробуйте `prices.iloc[2:5][0] = 999`. Сравните `.loc["a":"c"]` и `.iloc[0:3]`. Ожидаемый вывод: `prices.iloc[10:20]` содержит 10 элементов (конечная позиция не включается); изменение `subset` не влияет на `prices` (CoW); `prices.iloc[2:5][0] = 999` вызывает `ChainedAssignmentError`. В комментариях объясните: почему в Pandas 3.0 chained assignment не работает (CoW запрещает неявные цепочки); в каких случаях `.copy()` всё ещё нужен (если нужно гарантированно независимое изменение); как `np.shares_memory` помогает проверить view; почему `.loc` включает конечную метку, а `.iloc` — нет; что произойдёт при `.iloc` с шагом 0 (`ValueError`); как срез с отрицательным шагом влияет на порядок индекса.


In [ ]:
prices = pd.Series(np.arange(30) * 1000 + 5000)
print(prices.iloc[:7].tolist())
print(prices.iloc[-7:].tolist())
print(len(prices.iloc[10:20]))              # 10 — конечная позиция не включается
print(prices.iloc[::2].tolist())
print(prices.iloc[::-1].tolist())

subset = prices.iloc[2:5]
subset.iloc[0] = 999
print(prices.iloc[2])                       # не изменилось (CoW)

try:
    prices.iloc[2:5][0] = 999
except Exception as e:
    print(type(e).__name__, e)
print()

# Pandas 3.0: chained assignment запрещён (CoW) -> ChainedAssignmentError
# .copy() всё ещё нужен, когда хочешь гарантированно независимую копию
# np.shares_memory(a, b) -> True, если общий буфер (view)
# .loc включает правую метку, .iloc — нет
# .iloc шаг 0 -> ValueError: slice step cannot be zero
# отрицательный шаг меняет порядок индекса на обратный
print()


**Задача 15 (Boolean indexing, query, isin/between/str.contains).** Создайте Series из 9 цен абонементов. Отфильтруйте: `> 20000`, `< 10000`, `10000–20000`, `< 8000 | > 40000` (каждое условие в скобках). Замените цены `< 5000` на `5000` через `.loc`. Сравните с `prices.where(prices >= 5000, 5000)`. Примените `.query("price > 10000 and price < 35000")`, `.query("hall == 'бассейн'")`. Проверьте `.isin(["йога", "кроссфит"])`, `.between(10000, 20000)`, `.str.contains("Абонемент", na=False)`. Ожидаемый вывод: `prices[prices > 20000]` содержит подходящие значения; `prices.where(prices >= 5000, 5000)` заменяет `4500` на `5000`; `.query` и булева маска дают одинаковый результат. В комментариях объясните: почему `and`/`or` не работают с масками (нужны `&`/`|` с побитовой логикой); почему `na=False` критичен для `str.contains`; чем `.query()` удобнее булевой маски (читаемее, меньше скобок); почему каждое условие в `&`/`|` должно быть в скобках (приоритет операторов); когда `where` удобнее `.loc` (короткая замена с сохранением формы); как `@` в `query` ссылается на переменные окружения (`@min_price`).


In [ ]:
prices = pd.Series([12000, 24000, 8000, 36000, 6000, 18000, 48000, 4000, 60000])
print(prices[prices > 20000].tolist())
print(prices[prices < 10000].tolist())
print(prices[(prices >= 10000) & (prices <= 20000)].tolist())
print(prices[(prices < 5000) | (prices > 40000)].tolist())

prices.loc[prices < 5000] = 5000
print(prices.tolist())
print(prices.where(prices >= 5000, 5000).tolist())

df = pd.DataFrame({"price": [12000, 24000, 8000],
                   "hall": ["йога", "бассейн", "кроссфит"]})
print(df.query("price > 5000 and price < 20000"))
print(df.query("hall == 'бассейн'"))
print(df["hall"].isin(["йога", "кроссфит"]).tolist())
print(df["price"].between(10000, 20000).tolist())
print()

# and/or не работают поэлементно -> ValueError: truth value is ambiguous
# na=False в str.contains: на NaN вернёт False, а не NaN
# .query удобнее: меньше скобок, читается как SQL
# каждое условие в &/| — в скобках (приоритет сравнений выше &/|)
# where удобнее .loc, когда нужно вернуть новый объект той же формы
# var в query подставляет переменную из окружения
print()


**Задача 16 (where/mask/replace/map).** Создайте Series цен абонементов. Примените `prices.where(prices >= 5000, 5000)` и `prices.mask(prices > 40000, 40000)`. Создайте Series статусов `["активный", "заморожен", "истёк"]` и замените через `replace({"активный": "А", "заморожен": "З"})`. Сравните с `map({"активный": "А", "заморожен": "З"})`. Ожидаемый вывод: `where` заменяет там, где условие **False**; `mask` — там, где **True**; `replace` оставляет `"истёк"` без изменений; `map` возвращает `NaN` на `"истёк"`. В комментариях объясните: в каких случаях `where` удобнее `.loc` (когда нужно заменить все несоответствующие, а не точечно); почему `map` возвращает `NaN` для отсутствующих ключей, а `replace` — нет; как `mask` заменяет там, где True; чем `replace` со словарём отличается от `replace` со списками (списки — позиционное соответствие); что вернёт `map(lambda x: ...)` для строкового столбца (применяет функцию к каждому значению); как `where`/`mask` обрабатывают пропуски (по умолчанию сохраняют `NaN`, можно указать `other=`).

**Задача 17 (cut/qcut/crosstab/pivot_table).** Создайте DataFrame `clients` с `age` (14–80), `city`, `gender`, `spend_amount`. Примените `pd.cut(age, bins=[0, 18, 30, 50, 100], labels=["<18", "18-30", "30-50", ">50"])` и `pd.qcut(age, q=4, labels=["Q1", "Q2", "Q3", "Q4"])`. Сравните размеры групп через `value_counts()`. Создайте `crosstab(index="city", columns="age_group", margins=True, normalize="index")` и `pivot_table(values="spend_amount", index="city", columns="gender", aggfunc="mean", margins=True, fill_value=0)`. Ожидаемый вывод: `cut` даёт группы разной ширины (разного размера), `qcut` — примерно равного; `margins=True` добавляет строку `All`; `normalize="index"` показывает доли по строкам. В комментариях объясните: в каких случаях `qcut` падает (дубликаты в данных) и как решается через `duplicates="drop"`; как `normalize` меняет интерпретацию (`"index"` — доли по строкам, `"columns"` — по столбцам, `True` — по всей таблице); как `margins` добавляет итоги; чем `pivot_table` безопаснее `pivot` (умеет агрегировать дубликаты через `aggfunc`); как `fill_value` влияет на результат (заменяет `NaN` после агрегации); что произойдёт при `cut` с `right=False` (границы не включаются справа).

**Задача 18 (NaN: isna/dropna/fillna).** Создайте Series с пропусками (email клиентов). Проверьте `isna()`, `notna()`, `isna().sum()`. Примените `dropna(how="all")`, `dropna(thresh=2)`, `dropna(subset=["email"])`. Заполните через `fillna(0)`, `fillna(df.mean())`, `fillna(df.median())`, `fillna(df.mode().iloc[0])`. Для DataFrame примените `fillna({"A": 0, "B": df["B"].mean()})`. Проверьте, что `series == np.nan` всегда `False`. Ожидаемый вывод: `dropna(how="all")` удаляет только строки, где все значения `NaN`; `thresh=2` оставляет строки с ≥2 не-пропусками; `fillna(mode)` для строкового столбца возвращает моду (самое частое значение). В комментариях объясните: чем `thresh` отличается от `how` (`thresh` — минимальное число не-пропусков, `how` — "any"/"all"); почему `fillna(method=...)` удалён в Pandas 3.0 (используйте `ffill()`/`bfill()`); как `dropna(ignore_index=True)` сбрасывает индекс; почему сравнение с `np.nan` не работает (`NaN != NaN` по IEEE 754); в каких случаях `fillna(mode)` предпочтительнее `mean` (для категориальных/строковых данных, например, город); как `subset` ограничивает область проверки (только указанные столбцы).

**Задача 19 (ffill/bfill/interpolate).** Создайте Series с пропусками (дневная посещаемость клуба). Примените `ffill()`, `bfill()`, `ffill(limit=1)`. Создайте Series с `DatetimeIndex` и пропусками. Примените `interpolate(method="linear")`, `method="time"`, `method="polynomial", order=2`. Сравните результаты на неравномерном индексе. Примените `interpolate(limit_direction="both")`. Ожидаемый вывод: `ffill` копирует предыдущее значение, `bfill` — следующее; `linear` и `time` дают разные результаты на неравномерном индексе; `polynomial` может дать выбросы за пределы данных. В комментариях объясните: в каких случаях `time` лучше `linear` (когда интервалы между тренировками неравномерны — праздники, выходные); почему `polynomial` может дать выбросы (феномен Рунге); как `limit_direction` влияет на заполнение с краёв (`"forward"`, `"backward"`, `"both"`); чем `ffill` отличается от `interpolate` по семантике (копирование vs аппроксимация); что произойдёт при `interpolate` на нечисловом столбце (`TypeError` или игнорирование); как `method="nearest"` работает на категориальных данных.

**Задача 20 (duplicated/drop_duplicates/replace).** Создайте DataFrame посещений с дубликатами. Проверьте `duplicated()`, `duplicated(subset=["visit_id"])`, `duplicated(keep=False)`. Удалите через `drop_duplicates()`, `keep="last"`, `keep=False`, `subset=["visit_id"]`. Примените `replace(1, 100)`, `replace([1, 2], [100, 200])`, `replace({"старый": "новый"})`. Ожидаемый вывод: `duplicated()` возвращает `[False, False, True, ...]`; `keep=False` помечает все вхождения; `drop_duplicates(subset=["visit_id"])` удаляет по ключу. В комментариях объясните: чем `keep=False` отличается от `keep="first"` (`False` удаляет все дубликаты, `"first"` оставляет первый); в каких случаях `subset` важнее полного сравнения строк (когда есть служебные столбцы с датой/временем посещения); как `replace` работает с `regex=True` (регулярные выражения); что произойдёт при `replace` значения, отсутствующего в данных (ничего, без ошибки); как `replace` со словарём обрабатывает отсутствующие ключи (игнорирует); чем `replace` отличается от `map` (`map` заменяет все значения, отсутствующие → `NaN`; `replace` — только указанные).

**Задача 21 (astype/to_numeric/to_datetime).** Создайте Series `["1", "2", "abc", "4", "5"]` (количество тренировок в неделю). Примените `astype(int)` — зафиксируйте `ValueError`. Примените `pd.to_numeric(s, errors="coerce")`. Создайте Series `["2025-01-01", "invalid", "2025-02-01"]`, примените `pd.to_datetime(errors="coerce")`. Примените `pd.to_numeric(col, downcast="integer")`. Примените `df.convert_dtypes()`. Ожидаемый вывод: `astype(int)` на `"abc"` даёт `ValueError`; `to_numeric(errors="coerce")` даёт `NaN` на `"abc"`; `to_datetime(errors="coerce")` даёт `NaT` на `"invalid"`; `downcast` уменьшает тип. В комментариях объясните: почему `errors="ignore"` удалён в Pandas 3.0 (непредсказуемое поведение); чем `Int64` лучше `int64` при наличии `NaN` (поддерживает `pd.NA`); как `downcast` выбирает тип (минимальный, вмещающий все значения); в каких случаях `convert_dtypes` даёт неожиданный результат (может превратить строки в `string` dtype); что произойдёт при `to_datetime` с неверным форматом без `errors` (`ValueError`); как `to_numeric` обрабатывает строки с пробелами (обычно обрезает, но лучше `str.strip()`).

**Задача 22 (базовый .str).** Создайте Series названий абонементов. Примените `.str.lower()`, `.str.upper()`, `.str.title()`, `.str.strip()`, `.str.len()`, `.str.contains("abc", na=False)`, `.str.startswith("Абонемент")`, `.str.endswith("1")`, `.str.fullmatch(r"\d{4}")`. Проверьте `is_string_dtype()` перед применением. Попробуйте `.str.lower()` на числовом столбце. Ожидаемый вывод: `.str.len()` возвращает длину каждой строки; `.str.contains("abc", na=False)` на `NaN` возвращает `False`; `.str.lower()` на числовом — `AttributeError`. В комментариях объясните: почему `.str` не работает с числовым столбцом (нужен `.astype(str)`); почему `na=False` критичен (иначе `NaN` пропагируется); чем `fullmatch` отличается от `contains` (полное совпадение vs подстрока); как `.str.len()` обрабатывает `NaN` (возвращает `NaN`); в каких случаях `casefold` предпочтительнее `lower` (для Unicode, например, немецкая ß); что произойдёт при `.str` к `category`-столбцу (работает, если категории строковые, иначе `AttributeError`).

---



## 🟣 Уровень 3. Мастер (задачи 23–32)

**Задача 23 (.str.split/extract).** Создайте Series имён клиентов `"Анна Иванова"`. Примените `.str.split(expand=True)` — получите DataFrame с двумя столбцами. Примените `.str.split("-", n=1, expand=True)` к строкам с дефисом (например, "Иванов-Петров") и `.str.rsplit("-", n=1, expand=True)`. Создайте Series дат посещений `"2025-01-15"`. Извлеките год, месяц, день через `.str.extract(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})")`. Примените `.str.extractall(r"(\d)")` и `.str.findall(r"\d+")`. Ожидаемый вывод: `split(expand=True)` возвращает DataFrame 10×2; `extract` с именованными группами возвращает DataFrame с колонками `year`, `month`, `day`; `extractall` возвращает MultiIndex. В комментариях объясните: зачем `expand=True` (возвращает DataFrame вместо Series списков); как именованные группы делают код читаемее; чем `extract` отличается от `extractall` (первое совпадение vs все совпадения); как `n=1` ограничивает разделения (разделить только один раз); что произойдёт при `extract` без групп (возвращает одну колонку с полным совпадением); как `findall` обрабатывает несколько совпадений (возвращает список).


In [ ]:
names = pd.Series(["Анна Иванова", "Пётр Сидоров", "Иванов-Петров", "Мария Кузнецова"])
print(names.str.split(expand=True))
print(names.str.split("-", n=1, expand=True))
print(names.str.rsplit("-", n=1, expand=True))

dates = pd.Series(["2025-01-15", "2025-02-20"])
print(dates.str.extract(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})"))
print(dates.str.extractall(r"(\d)").head())
print(dates.str.findall(r"\d+").tolist())
print()

# expand=True -> DataFrame вместо Series списков
# именованные группы (?P<name>...) делают колонки читаемыми
# extract — первое совпадение; extractall — все (MultiIndex)
# n=1 ограничивает число разделений
# extract без групп -> одна колонка с полным совпадением
# findall -> список всех совпадений
print()


**Задача 24 (.str.replace/cat/pad/zfill).** Создайте Series email клиентов. Примените `.str.replace("@.*", "", regex=True)`. Сравните `.str.replace("f.", "ba", regex=False)` и `regex=True`. Примените `.str.replace({"apple": "груша", "banana": "банан"})`. Примените `.str.cat(series2, sep=" ")` и `.str.cat(series2, sep=" ", na_rep="?")`. Примените `.str.pad(10, side="left", fillchar="0")` и `.str.zfill(6)`. Ожидаемый вывод: `regex=True` даёт `"foo" → "bao"`, `regex=False` — без изменений; `na_rep="?"` заполняет пропуски; `zfill(6)` дополняет нулями слева. В комментариях объясните: почему `regex=True` нужно указывать явно (в Pandas 3.0 поведение по умолчанию изменено на `False`); как `na_rep` обрабатывает пропуски; чем `zfill` отличается от `pad` (`zfill` — только нули слева, `pad` — гибко); как `replace` со словарём работает с несколькими паттернами; что произойдёт при `replace` с неверным паттерном (`re.error`); как `isalpha` обрабатывает кириллицу (возвращает `True` для букв).


In [ ]:
emails = pd.Series(["anna@mail.ru", "petr@yandex.ru", None, "ivan@gmail.com"])
print(emails.str.replace(r"@.*", "", regex=True).tolist())

print(pd.Series(["foo", "foobar"]).str.replace("f.", "ba", regex=False).tolist())
print(pd.Series(["foo", "foobar"]).str.replace("f.", "ba", regex=True).tolist())

# словарь — через Series.replace(regex=True), а не str.replace
print(pd.Series(["apple", "banana"]).replace({"apple": "груша", "banana": "банан"}, regex=True).tolist())

s1 = pd.Series(["Анна", "Пётр"])
s2 = pd.Series(["Иванова", "Сидоров"])
print(s1.str.cat(s2, sep=" ").tolist())
print(pd.Series(["Анна", None]).str.cat(pd.Series(["Иванова", "Сидоров"]), sep=" ", na_rep="?").tolist())

print(pd.Series(["42", "7"]).str.pad(10, side="left", fillchar="0").tolist())
print(pd.Series(["42", "7"]).str.zfill(6).tolist())
print()

# str.replace требует явный repl: .str.replace(pat, repl, regex=...)
# словарь в str.replace не поддерживается в старых Pandas -> TypeError
# для замены по словарю используем Series.replace(dict, regex=True)
# na_rep="?" заполняет пропуски при cat
# zfill(6) — только нули слева; pad — гибко
# неверный regex -> re.error
print()


**Задача 25 (groupby: size/count/agg).** Создайте DataFrame `visits` с `hall`, `price` (с пропусками), `visits_count`. Примените `groupby("hall").size()` и `.count()` — сравните. Примените `sum()`, `mean()`, `first()`, `last()`, `nunique()`. Проверьте `skipna=False`. Примените `.agg(["sum", "mean"])` и `.agg(total=("price", "sum"), avg=("price", "mean"))`. Ожидаемый вывод: `size()` считает все строки, `count()` — только не-пропуски по каждому столбцу; `skipna=False` даёт `NaN` для групп с пропусками; `NamedAgg` возвращает именованные столбцы. В комментариях объясните: чем `size()` отличается от `count()` (включает `NaN`); как `skipna=False` влияет; почему `NamedAgg` читаемее списка; как `first`/`last` обрабатывают пропуски (по умолчанию пропускают, `skipna=False` — нет); что произойдёт при `agg` с несуществующей функцией (`AttributeError`); как `nunique` считает уникальные с `NaN` (по умолчанию исключает, `dropna=False` — включает).



**Задача 26 (groupby: transform/filter/apply).** Используйте DataFrame из задачи 25. Примените `.agg({"price": ["sum", "mean"], "visits_count": ["count", "max"]})`. Примените `transform("mean")` и `transform(lambda x: x.fillna(x.mean()))`. Примените `filter(lambda g: len(g) > 50)`. Примените `apply(weighted_avg)` с `np.average(price, weights=visits_count)`. Примените `apply(..., include_groups=False)`. Ожидаемый вывод: `transform("mean")` возвращает Series той же длины; `filter` оставляет только группы с >50 строками; `apply` с `include_groups=False` не передаёт группировочные столбцы в функцию. В комментариях объясните: в каких случаях `transform` незаменим (когда нужен результат той же длины, например, заполнение пропусков групповым средним); почему `apply` медленнее встроенных методов; что делает `include_groups=False` (убирает группировочные столбцы из DataFrame внутри функции); как `filter` отличается от булевой маски (фильтрует целые группы, а не строки); когда `agg` со словарём предпочтительнее списка (разные функции для разных столбцов); почему `transform` возвращает Series той же длины, а `agg` — свёрнутый результат.

**Задача 27 (groupby: multi-column/as_index/observed).** Создайте DataFrame с `city`, `hall`, `visits_count`. Примените `groupby(["city", "hall"]).size()`. Примените `groupby("city", as_index=False).agg(...)`. Проверьте `observed=True/False` для `category`-столбца. Проверьте `dropna=False` и `sort=False`. Итерируйте через `for name, group in df.groupby("city"): ...`. Используйте `df.groupby("city").get_group("Москва")`. Ожидаемый вывод: `as_index=False` возвращает `city` как столбец; `observed=False` включает ненаблюдаемые категории; `get_group("Москва")` возвращает подмножество. В комментариях объясните: когда `as_index=False` удобнее (для дальнейшего `merge`/`query` без `reset_index`); как `observed` влияет на результат (с `False` — все категории, с `True` — только встречающиеся); почему `get_group` быстрее `filter` (прямой доступ к группе, без обхода всех); как `dropna=False` включает `NaN`-группы; что произойдёт при `get_group` с несуществующей группой (`KeyError`); как `sort=False` ускоряет группировку (не сортирует ключи).

**Задача 28 (cut/qcut/crosstab/pivot_table, продвинутый).** Создайте DataFrame `clients` с `age`, `city`, `gender`, `spend_amount`. Примените `cut` и `qcut` к `age`. Создайте `crosstab(index="city", columns="age_group", margins=True, normalize="index")` и `crosstab(..., values="spend_amount", aggfunc="sum")`. Создайте `pivot_table(values="spend_amount", index="city", columns="gender", aggfunc=["sum", "mean"], margins=True, fill_value=0)`. Примените `cut` с `right=False`. Ожидаемый вывод: `aggfunc=["sum", "mean"]` создаёт MultiIndex по столбцам; `right=False` сдвигает границы; `fill_value=0` заменяет `NaN`. В комментариях объясните: как `cut` с `right=False` меняет границы (интервалы `[a, b)` вместо `(a, b]`); когда `margins` полезен (итоговые строки/столбцы); как `aggfunc` списком влияет на структуру (MultiIndex в столбцах); чем `crosstab` с `values` отличается от `pivot_table` (`crosstab` — частотная таблица, но с `values`/`aggfunc` становится агрегацией); как `normalize="columns"` меняет интерпретацию (доли по столбцам); что произойдёт при `qcut` с `q=4` на малых данных (может упасть из-за дубликатов или недостатка значений).

**Задача 29 (rolling).** Создайте временной ряд из 100 дней (посещаемость). Примените `rolling(7).mean()`, `rolling(7, min_periods=1).mean()`, `rolling(7, center=True).mean()`, `rolling(7, win_type="triang").mean()`, `rolling("30D").sum()`, `rolling("5B").mean()`. Примените `.agg(["mean", "std"])` и `.apply(np.percentile(x, 90), raw=True)`. Ожидаемый вывод: `rolling(7).mean()` — первые 6 значений `NaN`; `min_periods=1` — без `NaN`; `center=True` — метка в центре окна; `rolling("30D")` требует `DatetimeIndex`. В комментариях объясните: чем `center=True` отличается (окно центрируется на текущей точке, а не заканчивается в ней); когда `raw=True` ускоряет `apply` (передаёт NumPy-массив вместо Series); как `win_type` меняет веса (треугольное, гауссово окно); почему временные окна требуют `DatetimeIndex`; что произойдёт при `min_periods > window` (все значения `NaN`); как `closed` влияет на включение границ (`"right"`, `"left"`, `"both"`, `"neither"`).

**Задача 30 (expanding/ewm/groupby().rolling()).** Примените `expanding().mean()` и `expanding().sum()`. Примените `ewm(span=10).mean()`, `ewm(halflife=5).mean()`, `ewm(alpha=0.3).mean()`, `ewm(span=10, adjust=False).mean()`. Сравните влияние параметров. Примените `groupby("hall")["visits_count"].transform(lambda x: x.rolling(7).mean())`. Ожидаемый вывод: `expanding` даёт накопительное среднее; `span=10` соответствует `α=2/11≈0.18`; `adjust=False` даёт другую формулу (рекурсивную); `groupby().rolling()` возвращает Series с исходным индексом через `transform`. В комментариях объясните: чем `expanding` отличается от `rolling` (накопительное окно от начала vs скользящее фиксированное); как `adjust=False` меняет формулу (рекурсивное взвешивание без деления на сумму весов); почему `transform` нужен для `groupby().rolling()` (чтобы вернуть исходную форму); как `halflife` связан с `span` (`halflife = ln(2)/α`, `span = 2/α - 1`); что произойдёт при `ewm` без параметров (`span=1` по умолчанию); как `ignore_na=True` влияет на результат (пропуски не влияют на веса).

**Задача 31 (concat/stack/unstack/melt).** Создайте два DataFrame посещений за разные периоды с одинаковыми столбцами. Примените `pd.concat([df1, df2])`, `pd.concat(..., ignore_index=True)`, `pd.concat(..., axis=1)`, `pd.concat(..., keys=["a", "b"])`. Попробуйте `pd.concat(..., ignore_index=True, keys=["a", "b"])`. Примените `df.stack()`, `df.unstack()`, `pd.melt(df, id_vars=["hall"], value_vars=["Q1", "Q2"], var_name="quarter", value_name="revenue")`. Ожидаемый вывод: `concat` без `ignore_index` дублирует индексы; с `ignore_index` — пересоздаёт; `keys` создаёт MultiIndex; `ignore_index + keys` даёт `ValueError` в Pandas 3.0. В комментариях объясните: почему `concat(ignore_index=True, keys=...)` даёт ошибку в 3.0 (конфликт семантики); чем `melt` отличается от `stack` (`melt` — wide-to-long по столбцам, `stack` — по индексу); как `stack(dropna=False)` сохраняет пропуски; что произойдёт при `unstack` с несуществующим уровнем (`KeyError`); как `keys` создаёт MultiIndex; в каких случаях `melt` предпочтительнее `stack` (когда нужно переплавить широкую таблицу с несколькими value_vars).

**Задача 32 (merge/join/anti-join).** Создайте `visits` и `clients`. Примените `merge(how="inner")`, `"left"`, `"right"`, `"outer"`, `"left_anti"`, `"right_anti"`. Используйте `on="client_id"`, `left_on`/`right_on`, `suffixes=("_v", "_c")`, `indicator=True`. Проверьте `validate="one_to_one"` — зафиксируйте `MergeError` при дубликатах. Создайте `visits.set_index("client_id").join(clients.set_index("client_id"))`. Ожидаемый вывод: `inner` оставляет только совпадающие ключи; `left_anti` возвращает посещения без клиентов; `indicator=True` добавляет столбец `_merge`; `validate="one_to_one"` вызывает `MergeError` при дубликатах. В комментариях объясните: чем `merge` отличается от `join` (`join` — по индексу, `merge` — по столбцам); когда anti-join полезен (поиск «осиротевших» посещений без клиентов); что делает `indicator=True` (добавляет `_merge` с `"both"`/`"left_only"`/`"right_only"`); как `suffixes` предотвращает конфликт имён; почему `validate` защищает от декартова произведения; как `merge` обрабатывает `NaN` в ключах (не совпадают, остаются в `left_only`/`right_only`).

---



## 🟠 Уровень 4. Архитектор (задачи 33–42)

**Задача 33 (Timestamp/Timedelta/Period).** Создайте `pd.Timestamp("2025-01-15 14:30")` (время тренировки) — извлеките `year`, `month`, `day`, `hour`, `dayofweek`, `day_name()`, `is_month_end`. Прибавьте `pd.Timedelta(days=5)`. Создайте `pd.Timedelta("3 days 2 hours")` — выведите `total_seconds()`. Создайте `pd.Period("2025-01", freq="M")` — выведите `start_time`, `end_time`, прибавьте `3`. Примените `pd.to_datetime(["2025-01-01", "invalid"], errors="coerce")`. Ожидаемый вывод: `Timestamp("2025-01-15").day_name()` = `"Wednesday"`; `Timedelta("3 days 2 hours").total_seconds()` = `266400.0`; `Period("2025-01") + 3` = `Period("2025-04", "M")`; `to_datetime(errors="coerce")` даёт `[Timestamp, NaT]`. В комментариях объясните: чем `Period` отличается от `Timestamp` (период vs момент); почему `errors="coerce"` безопаснее `raise` (не ломает пайплайн); чем `Timedelta` отличается от `Period` (длительность vs календарный период); как `Period` обрабатывает границы месяца (start_time/end_time); что произойдёт при `to_datetime` с неверным форматом без `errors` (`ValueError`); как `Timestamp` работает с часовыми поясами (`tz_localize`/`tz_convert`).



**Задача 34 (date_range, DatetimeIndex, .dt).** Создайте `pd.date_range("2024-01-01", periods=365, freq="D")`. Создайте диапазоны с `freq="B"`, `"ME"`, `"YE"`, `"h"`. Установите `DatetimeIndex` как индекс. Выполните `df["2024"]`, `df["2024-01"]`, `df["2024-01-15"]`, `df["2024-01-01":"2024-01-31"]`. Примените `.dt.year`, `.dt.month`, `.dt.day_name()`, `.dt.to_period("M")`. Ожидаемый вывод: `freq="B"` даёт 260 рабочих дней в году; `freq="ME"` — конец каждого месяца; `df["2024-01"]` возвращает 31 строку; `.dt.day_name()` возвращает день недели. В комментариях объясните: почему `freq="ME"` вместо `"M"` в Pandas 3.0 (унификация с `"YE"`, `"QE"`); как `df["2024-01"]` работает без `.loc` (частичная индексация по `DatetimeIndex`); когда `.dt` незаменим (извлечение компонент даты); чем частичная индексация отличается от `.loc` (более лаконичный синтаксис); что произойдёт при `df["2024-13"]` (`KeyError` или пустой результат); как `date_range` обрабатывает високосные годы (29 февраля включается при `freq="D"`).

**Задача 35 (resample/shift/diff/pct_change).** Создайте DataFrame с `DatetimeIndex` (2 года ежедневной посещаемости). Примените `resample("ME").agg(total=("visits_count", "sum"), avg=("visits_count", "mean"))`, `resample("QE").size()`, `resample("D").ffill()` (upsampling). Примените `shift(1)`, `shift(-1)`, `diff()`, `pct_change() * 100`. Примените `groupby("client_id").transform(lambda x: x.rolling(7).sum())`. Ожидаемый вывод: `resample("ME").agg` даёт 24 строки; `shift(1)` сдвигает значения, `shift(-1)` — в обратную сторону; `diff()` даёт разности, `pct_change()` — проценты; `ffill()` при upsampling копирует значение вперёд. В комментариях объясните: чем downsampling отличается от upsampling (агрегация vs интерполяция/заполнение); когда `ffill` лучше `interpolate` (для категориальных/дискретных данных); как `pct_change` используется в финансах (рост выручки клуба); почему `shift` с `freq` сдвигает индекс, а не данные (календарный сдвиг); что произойдёт при `resample` без `DatetimeIndex` (`TypeError`); как `resample` связан с `groupby` (по сути — группировка по временным интервалам).

**Задача 36 (tz_localize/tz_convert).** Создайте `DatetimeIndex` без пояса (время тренировки). Примените `tz_localize("UTC")` и `tz_convert("Europe/Moscow")`. Убедитесь, что моменты совпадают (`ts_utc == ts_msk`). Создайте `pd.date_range("2024-01-01", periods=3, freq="D", tz="UTC")`. Используйте `zoneinfo.available_timezones()`. Попробуйте `tz_convert` на naive-данных. Ожидаемый вывод: `tz_localize("UTC")` добавляет `+00:00`; `tz_convert("Europe/Moscow")` сдвигает на `+03:00`; сравнение `ts_utc == ts_msk` — `True`; `tz_convert` на naive — `TypeError`. В комментариях объясните: почему `tz_convert` нельзя на naive-данных (неясно, из какой зоны конвертировать); чем `tz_localize` отличается от `tz_convert` (назначить зону vs пересчитать в другую); почему в 3.0 используется `zoneinfo` (стандартная библиотека вместо `pytz`); что произойдёт при `tz_localize` в зоне с DST (неоднозначное/несуществующее время); как `ambiguous` и `nonexistent` обрабатывают переходы; как сравнить два tz-aware Timestamp (автоматически приводятся к UTC).

**Задача 37 (categorical: создание, коды, ordered).** Создайте `pd.Series(["a", "b", "c", "a"], dtype="category")`. Примените `astype("category")` к строковому столбцу. Создайте `pd.Categorical(["средний", "низкий", "высокий"], categories=["низкий", "средний", "высокий"], ordered=True)`. Проверьте `.cat.categories`, `.cat.codes`, `.cat.ordered`. Создайте `CategoricalDtype` с `ordered=True`. Отсортируйте Series. Примените `groupby("category", observed=False/True)`. Ожидаемый вывод: `codes` для `["a", "b", "c", "a"]` — `[0, 1, 2, 0]`; `ordered=True` даёт сортировку по порядку категорий; `observed=False` включает ненаблюдаемые категории. В комментариях объясните: почему `-1` — код пропуска; как `ordered=True` влияет на сортировку (порядок задаётся `categories`); чем `observed=False` отличается (включает все категории в `groupby`); что произойдёт при присвоении значения вне категорий (`ValueError`); как `category` влияет на память (хранит коды + словарь); почему `category` не подходит для высококардинальных данных (словарь занимает много памяти).

**Задача 38 (categorical: .cat.* API, память).** Примените `.cat.add_categories(["d", "e"])`, `.cat.remove_categories(["a"])`, `.cat.remove_unused_categories()`, `.cat.rename_categories({"a": "x"})`, `.cat.reorder_categories(["c", "b", "a"], ordered=True)`. Создайте 1 000 000 строк с 4 залами — сравните `memory_usage(deep=True)` для `str` и `category`. Создайте 10 000 строк с 10 000 уникальных значений — сравните `str` vs `category`. Ожидаемый вывод: `category` для 4 значений экономит ≈96% памяти; для 10 000 значений `category` использует **больше** памяти, чем `str`. В комментариях объясните: когда `category` хуже `str` (высокая кардинальность — например, уникальные ФИО клиентов); как `rename_categories` со списком работает (позиционное соответствие); зачем `remove_unused_categories` (убирает категории, которых нет в данных); что произойдёт при `remove_categories` с используемым значением (`ValueError`); как `reorder_categories` влияет на `sort_values`; как `category` взаимодействует с `groupby` (эффективнее, чем `str`).

**Задача 39 (статистика, cov/corr).** Создайте Series цен абонементов. Вычислите `mean`, `median`, `std(ddof=0)`, `std(ddof=1)`, `var`, `quantile([0.25, 0.5, 0.75])`. Создайте DataFrame метрик 365×5 (показатели клуба: посещаемость, выручка, число клиентов, средний BMI, загрузка залов). Вычислите `cov()`, `corr()`, `corr(method="spearman")`. Проверьте симметричность `corr` через `(corr == corr.T).all().all()`. Ожидаемый вывод: `std(ddof=1)` больше `std(ddof=0)`; `corr` симметрична, диагональ — единицы; `corr(method="spearman")` даёт ранговую корреляцию. В комментариях объясните: когда `ddof=1` (выборочное стандартное отклонение, по умолчанию в Pandas); почему медиана устойчива к выбросам (50-й перцентиль); чем `cov` отличается от `corr` (масштаб vs нормировка); как `spearman` обрабатывает нелинейные зависимости; что произойдёт при `corr` с `NaN` (по умолчанию pairwise-исключение); как `quantile` интерполирует между точками (по умолчанию линейно).

**Задача 40 (линейная алгебра).** Решите систему `A @ x = b` через `np.linalg.solve(A, b)` и через `np.linalg.inv(A) @ b`. Сравните результаты. Вычислите `np.linalg.det(A)`, `np.linalg.eig(A)`, `np.linalg.norm(b)`, `norm(b, ord=1)`, `norm(b, ord=np.inf)`. Оформите в `pd.Series`. Ожидаемый вывод: `solve` и `inv(A) @ b` дают одинаковый `x` (с точностью до плавающей погрешности); `det(A)` — скаляр; `eig(A)` возвращает `(eigenvalues, eigenvectors)`; `norm(b)` — евклидова норма. В комментариях объясните: почему `solve` устойчивее `inv` (меньше численных ошибок, не вычисляет обратную матрицу явно); что такое собственные значения (масштаб преобразования); когда матрица не имеет обратной (`det(A) == 0`); как норма используется в рекомендательных системах (расстояние между векторами предпочтений клиентов); как сохранить результат в Series с осмысленным индексом; что произойдёт при `solve` с вырожденной матрицей (`LinAlgError`).

**Задача 41 (сортировка: sort/rank/nlargest/partition/searchsorted/lexsort).** Создайте Series цен абонементов. Отсортируйте `sort_values()`, `sort_values(ascending=False)`. Примените `rank(ascending=False, method="min")`, `rank(method="dense")`. Примените `nlargest(3)`, `nsmallest(3)`. Создайте 1 000 000 значений — сравните `sort_values().tail(10)`, `nlargest(10)`, `np.partition(values, -10)[-10:]` через `time.perf_counter()`. Примените `np.searchsorted(bounds, prices)`. Примените `np.lexsort((prices, halls))`. Ожидаемый вывод: `rank(method="min")` для `[100, 100, 90]` — `[1, 1, 3]`; `partition` в 10–50 раз быстрее `sort` на 1M; `searchsorted` возвращает позицию для вставки. В комментариях объясните: почему `partition` O(n) (не сортирует полностью, только разделяет); как `searchsorted` с `side` обрабатывает границы (`"left"`/`"right"`); почему последний ключ `lexsort` — основной (сортировка по последнему, затем по предпоследнему); чем `rank` отличается от `argsort` (`rank` — позиция в отсортированном ряду, `argsort` — индексы); что произойдёт при `nlargest` с `keep="all"` (все равные максимальные); как `sort_values` обрабатывает `NaN` (по умолчанию в конец).

**Задача 42 (unique/isin/set ops/duplicated).** Создайте два Series залов. Примените `.unique()` и `np.unique()` — сравните сортировку. Примените `.value_counts()`, `.value_counts(normalize=True)`, `.value_counts(dropna=False)`. Используйте `np.intersect1d`, `np.union1d`, `np.setdiff1d`, `np.setxor1d`, `.isin()`. Примените `duplicated()`, `drop_duplicates()`. Ожидаемый вывод: `np.unique` сортирует, `.unique()` — нет (порядок появления); `value_counts(normalize=True)` возвращает доли; `setdiff1d` возвращает элементы первого массива, отсутствующие во втором. В комментариях объясните: почему `np.unique` сортирует (реализация через сортировку); когда `isin` удобнее `set` (векторизация, сохранение порядка); как `value_counts(dropna=False)` считает `NaN`; чем `drop_duplicates` отличается от `duplicated` (удаляет vs помечает); что произойдёт при `setxor1d` с одинаковыми массивами (пустой массив); как `value_counts` сортирует результат (по убыванию частоты).

---

## 🔴 Уровень 5. Эксперт (задачи 43–52)

**Задача 43 (np.where/select/clip).** Примените `np.where(prices < 5000, 5000, prices)`, `np.select([prices < 10000, prices < 25000, prices >= 25000], ["базовый", "стандарт", "премиум"], default="?")`, `np.clip(prices, 5000, 45000)`. Сравните с `prices.where(prices >= 5000, 5000)` и `prices.mask(prices > 40000, 40000)`. Примените `np.log`, `np.sqrt`, `np.abs`, `np.round`, `np.cumsum`, `np.percentile`. Ожидаемый вывод: `np.where` и `.where` дают одинаковый результат; `np.select` возвращает `["базовый", "стандарт", "премиум"]` по порядку; `np.clip` обрезает с обеих сторон; `np.log(0)` — `-inf`. В комментариях объясните: почему порядок условий в `select` важен (первое совпадение); когда `clip` лучше `where` (обрезать с двух сторон); чем `np.where` отличается от `.where()` (`np.where` возвращает NumPy-массив, `.where()` — Series); что вернёт `np.log` для 0 и отрицательных (`-inf`, `nan`); как `np.cumsum` связан с `expanding().sum()`; когда `np.select` предпочтительнее вложенного `np.where` (читаемость для 3+ условий).

**Задача 44 (broadcasting/axis/keepdims).** Создайте `attendance` 3×4 (дни × залы) и `offset = pd.Series([10, 20, 30, 40], index=attendance.columns)`. Сложите `attendance + offset`. Создайте `offset_col = pd.Series([5, 10, 15], index=attendance.index)` — сложите через `attendance.add(offset_col, axis=0)`. Попробуйте `attendance + offset_col` без `axis`. Примените `attendance.values - attendance.values.mean(axis=1, keepdims=True)` и без `keepdims`. Ожидаемый вывод: `attendance + offset` выравнивает по столбцам; `attendance + offset_col` без `axis` даёт `NaN`; `keepdims=True` даёт форму `(3, 1)`, без — `(3,)` с ошибкой broadcasting. В комментариях объясните: почему `(3,)` несовместима с `(3, 4)` без `axis` (нужно выравнивание по строкам); как `fill_value` в `.add()` помогает; когда `keepdims` экономит код (сохранение размерности для broadcasting); что произойдёт при `axis=2` (`ValueError`); как `np.newaxis` заменяет `keepdims`; как broadcasting работает с трёхмерными массивами.

**Задача 45 (to_numpy/values/NumPy).** Примените `df.to_numpy()`, `df.to_numpy(dtype="float32")`, `df.to_numpy(copy=True)`, `df.values`. Создайте DataFrame из NumPy с разнородными типами (числа + строки). Примените `df.select_dtypes(include=[np.number]).to_numpy()`. Создайте NumPy-массив из Series, передайте в `LinearRegression`. Ожидаемый вывод: `df.to_numpy()` на смешанном DataFrame даёт `object`; `select_dtypes(include=[np.number])` сохраняет числовые типы; `to_numpy(copy=True)` создаёт независимую копию. В комментариях объясните: почему `.values` устарел (может вернуть view или копию непредсказуемо); как `copy=False` делит память (изменения в массиве влияют на DataFrame); когда `select_dtypes` спасает (перед ML-моделью для прогноза оттока клиентов); что произойдёт при `.to_numpy` с пропусками (`NaN` или `pd.NA`); как `dtype` влияет на память; почему `to_numpy` на `category` возвращает `object` (раскодирует категории).

**Задача 46 (SQL ↔ Pandas).** Переведите на Pandas: `SELECT hall, COUNT(*) AS cnt, AVG(price) AS avg_price FROM visits INNER JOIN clients ON ... WHERE status = 'completed' GROUP BY hall HAVING COUNT(*) > 100 ORDER BY avg_price DESC LIMIT 10`. Реализуйте через method chaining: `merge` → `query` → `groupby().agg()` → `reset_index()` → `query` → `sort_values` → `head(10)`. Реализуйте `UNION ALL` через `pd.concat`, `UNION` через `drop_duplicates()`. Реализуйте `LAG` через `shift`, `RANK` через `rank(method="min")`. Ожидаемый вывод: результат совпадает с SQL-запросом; `LAG` = `shift(1)`; `RANK` = `rank(method="min")`; `UNION ALL` = `concat(ignore_index=True)`. В комментариях объясните: чем `size()` отличается от `count()`; как `HAVING` через `query` после `groupby`; как перевести `ROW_NUMBER` (`groupby().cumcount() + 1`); почему `reset_index` нужен перед `query` по агрегату; как `LAG` с `groupby` работает (`groupby().shift(1)`); когда `as_index=False` заменяет `reset_index`.

**Задача 47 (df.plot: line/bar/barh/hist).** Создайте DataFrame с временным рядом посещаемости. Постройте `df.plot(kind="line", figsize=(12, 6), title="...")`, `df.plot(kind="bar", x="hall", y="visits_count")`, `df.plot(kind="barh", ...)`, `df["age"].plot(kind="hist", bins=20)`. Примените `xlabel`, `ylabel`, `grid`, `color`, `alpha`, `legend`. Ожидаемый вывод: `line` рисует линию по индексу; `bar` — вертикальные столбцы; `barh` — горизонтальные; `hist` — распределение с `bins` интервалами. В комментариях объясните: когда `barh` удобнее `bar` (длинные названия залов); зачем `bins` в гистограмме (число интервалов); как `alpha` влияет на читаемость (прозрачность); что произойдёт при `plot` без `figsize` (размер по умолчанию); как `legend` управляет подписями; почему `hist` работает только с одним столбцом (распределение одной переменной).

**Задача 48 (df.plot: scatter/box/area/pie).** Примените `df.plot(kind="scatter", x="price", y="visits_count", alpha=0.5)`, `df.plot(kind="box")`, `clients.boxplot(column="age", by="city")`, `df.plot(kind="area", stacked=True)`, `status_counts.plot(kind="pie", autopct="%1.1f%%")`. Ожидаемый вывод: `scatter` показывает связь двух переменных; `box` — квартили и выбросы; `area(stacked=True)` — накопленные доли; `pie` — доли от целого. В комментариях объясните: когда `box` лучше `hist` (сравнение групп, выбросы); почему `pie` плохо работает с >7 категорий (нечитаемость); как `stacked=True` в `area` показывает доли; что произойдёт при `scatter` без `x`/`y` (`ValueError`); как `boxplot(by=...)` группирует данные; почему `pie` требует один столбец.

**Задача 49 (Matplotlib: subplots, стили, даты).** Создайте `fig, axes = plt.subplots(2, 2, figsize=(14, 10), layout="constrained")`. На каждом подграфике постройте свой тип через `df.plot(ax=axes[i, j])`. Примените `ax.set_title()`, `ax.set_xlabel()`, `ax.grid()`, `ax.axhline()`, `ax.annotate()`. Используйте `plt.style.use("seaborn-v0_8")`. Настройте `mdates.MonthLocator(interval=3)` и `DateFormatter("%b %Y")`. Сохраните через `fig.savefig("plot.png", dpi=300, bbox_inches="tight")`. Ожидаемый вывод: 2×2 сетка подграфиков; стиль `seaborn-v0_8`; даты на оси X в формате `"Jan 2024"`. В комментариях объясните: когда нужен Matplotlib вместо `df.plot` (тонкая настройка); как `layout="constrained"` убирает обрезку; зачем `dpi=300` (качество печати); как `ax.annotate` работает с `xytext` (смещение текста); что произойдёт при сохранении без `bbox_inches="tight"` (обрезанные подписи); как комбинировать `df.plot` и Seaborn на одной фигуре.

**Задача 50 (визуализация временных рядов).** Создайте ряд `daily` с дневной посещаемостью (например, 365 дней с `DatetimeIndex`). Постройте на одной оси два графика: сырые данные через `daily.plot(alpha=0.3, ax=ax)` и сглаженный ряд через `daily.rolling(30).mean().plot(color="red", linewidth=2, ax=ax)`. Добавьте `ax.legend(["Данные", "Скользящее среднее (30)"])`, `ax.grid()`, `ax.set_title()`. Отформатируйте ось X через `mdates.MonthLocator(interval=3)` и `DateFormatter("%b %Y")`. Ожидаемый вывод: две линии на одной оси — полупрозрачная сырая и красная сглаженная; легенда с двумя записями; даты в формате `"Jan 2024"`. В комментариях объясните: зачем на графике скользящее среднее (убрать шум, увидеть тренд); как `alpha` помогает видеть обе линии одновременно; как `rolling` сглаживает шум; почему `ax=ax` обязателен для наложения графиков; как выбрать окно скользящего среднего (7 — неделя, 30 — месяц); что произойдёт при `rolling` без `min_periods` в начале ряда (первые 29 значений `NaN`).

**Задача 51 (pd.col() в цепочках).** Создайте DataFrame. Постройте цепочку `df.assign(ratio=pd.col("occupied") / pd.col("capacity")).loc[pd.col("ratio") > 0.5].assign(double_occupied=pd.col("occupied") * 2).sort_values("ratio")`. Сравните с вариантом через `lambda`. Проверьте, что `pd.col()` работает в `.loc` и `.assign()`. Примените `.pipe(print)` внутри цепочки. Ожидаемый вывод: оба варианта дают идентичный результат; `pd.col()` работает в `.loc` и `.assign()`; `.pipe(print)` выводит промежуточный DataFrame. В комментариях объясните: почему `pd.col()` пока «начальная поддержка» (новый API, не все методы поддерживают); в каких случаях `lambda` всё ещё нужен (сложные выражения, вызовы внешних функций); как `pd.col()` читается в длинных цепочках (без `lambda x: x[...]`); что произойдёт при `pd.col()` со списком столбцов (не поддерживается); как `.pipe` помогает отлаживать (вставка `print`/`head` в цепочку); когда цепочка становится нечитаемой (более 5–7 шагов).

**Задача 52 (SQL + Pandas + viz).** Возьмите SQL-запрос из задачи 46, реализуйте в Pandas, постройте через `df.plot(kind="barh", x="hall", y="avg_price")`. Добавьте `ax.axvline(df["avg_price"].mean(), color="red", linestyle="--", label="Среднее")`. Примените `ax.set_xlabel("Средняя цена (руб.)")`, `ax.legend()`. Ожидаемый вывод: горизонтальные столбцы по залам; вертикальная линия среднего; легенда; подпись оси. В комментариях объясните: как аннотировать средний уровень (axvline); почему `barh` удобнее для длинных названий залов; как сохранить график с высоким разрешением (`dpi=300`); что произойдёт при `plot` без `figsize`; как добавить подписи значений на столбцы (`ax.bar_label`); как сравнить два среза на одном графике (`hue`/`stacked`).

---

## ⚫ Уровень 6. Гуру (задачи 53–60)

**Задача 53 (векторизация vs iterrows).** Создайте DataFrame из 100 000 строк с `price`, `members`. Реализуйте `revenue = price * members` через `iterrows()`, `itertuples()`, `apply(axis=1)`, векторизацию `df["price"] * df["members"]`. Замерьте время через `time.perf_counter()` (прогрев + 5 замеров). Сравните `np.where(df["bmi"] > 30, "риск", "норма")` с `df["bmi"].apply(lambda x: ...)`. Ожидаемый вывод: `iterrows` ≈2.8 с, `itertuples` ≈0.09 с, `apply` ≈1.5 с, векторизация ≈0.001 с; ускорение векторизации относительно `iterrows` ≈2500×. В комментариях объясните: почему `iterrows` — антипаттерн (создаёт Series на каждую строку); когда `itertuples` уместен (когда нужны именованные кортежи); почему `apply(axis=1)` медленнее векторизации (Python-цикл); как `np.where` заменяет `apply`; что такое `namedtuple` (возвращает `itertuples`); как измерить время корректно (прогрев, несколько прогонов, `perf_counter`).

**Задача 54 (chunksize/usecols/dtype).** Создайте CSV на 500 000 строк (логи посещений). Прочитайте: целиком; через `chunksize=100_000` с агрегацией суммы; через `usecols=["visit_id", "price"]`; через `dtype={"visit_id": "int32", "price": "float32"}`. Сравните время и память (`memory_usage(deep=True)`). Ожидаемый вывод: `chunksize` даёт пиковую память ≈1/5 полного чтения; `usecols` сокращает память пропорционально числу столбцов; `dtype` уменьшает память в 2 раза для чисел. В комментариях объясните: когда `chunksize` обязателен (файл не влезает в память); как `usecols` экономит память; почему `dtype` при чтении лучше `astype` после (не создаётся промежуточный `object`); как агрегировать по чанкам (суммировать результаты); что произойдёт при `chunksize` больше размера файла (один чанк); как `nrows` помогает при тестировании (быстрая проверка структуры).

**Задача 55 (PyArrow: dtype_backend).** Создайте CSV на 1 000 000 строк со строковым столбцом (названия залов). Прочитайте с `dtype_backend="pyarrow"` и без — сравните `dtypes`, время, память. Примените `.str.lower()` на `str` (PyArrow) vs `object`. Ожидаемый вывод: с `dtype_backend="pyarrow"` строки имеют `dtype=string[pyarrow]`; скорость `.str.lower()` в 5–6× выше; память в 2 раза меньше. В комментариях объясните: когда `dtype_backend="pyarrow"` даёт максимальный выигрыш (большие строковые столбцы с названиями залов); как `str`-dtype (3.0) связан с PyArrow (нативный строковый тип); почему без PyArrow fallback на `object` (медленные Python-строки); что произойдёт при `dtype_backend="pyarrow"` на числовых (тоже ускоряет); как PyArrow влияет на Parquet (нативное чтение/запись); как сравнить производительность корректно.

**Задача 56 (антипаттерны).** Создайте DataFrame. Примените `df.drop("col", axis=1, inplace=True)` — зафиксируйте `None`. Воссоздайте и примените `df = df.drop("col", axis=1)`. Попробуйте `.append(new_row)` — зафиксируйте `AttributeError`. Замените на сбор в список + `pd.concat(rows, ignore_index=True)`. Создайте `subset = df[df["price"] > 0]`, измените `subset.loc[subset.index[0], "status"] = "completed"`. Выполните `df["A"][df["B"] > 0] = 1` — зафиксируйте `ChainedAssignmentError`. Ожидаемый вывод: `inplace=True` возвращает `None`; `.append` даёт `AttributeError`; `ChainedAssignmentError` возникает при chained assignment; `subset` не изменяет `df` (CoW). В комментариях объясните: почему `inplace` ломает цепочки (возвращает `None`); как CoW делает `.copy()` ненужным (по умолчанию копирование при записи); почему `.append()` удалён (медленный, каждый раз копирует); в каких случаях `.copy()` всё ещё нужен (принудительное копирование перед изменением); как переписать цикл с `.append` на `pd.concat`; почему `inplace=True` не экономит память для методов, меняющих форму.

**Задача 57 (Dask/Polars/Vaex).** Создайте DataFrame 100 000 строк, сохраните в `huge.csv` и `huge.hdf5`. Для Dask: `dd.read_csv("huge.csv")`, `df.groupby("hall")["visits_count"].sum().compute()` — сравните время с Pandas. Для Polars: `pl.read_csv("huge.csv")`, `df.group_by("hall").agg(pl.col("visits_count").sum())` — зафиксируйте разницу синтаксиса и скорость. Для Vaex: `vaex.open("huge.hdf5")`, `df[df["visits_count"] > 10].groupby("hall", agg="sum")`. Ожидаемый вывод: Dask ленив — вычисления только при `.compute()`; Polars в 2–10× быстрее Pandas; Vaex не загружает файл в память. В комментариях объясните: когда Pandas перестаёт справляться (данные не влезают в RAM); чем Dask отличается от Polars (ленивые вычисления vs быстрый Rust-движок); зачем Vaex нужен memory-mapping (работа с файлом на диске); как `.compute()` запускает вычисления; как Polars использует многоядерность; что такое out-of-core (вычисления вне RAM).

**Задача 58 (Pandas 3.0).** Создайте `pd.Series(["a", "b", "c"])` — зафиксируйте `dtype="str"`. Примените `is_string_dtype(series)` — `True`; сравните с `series.dtype == object` — `False`. Выполните `df["A"][df["B"] > 0] = 1` — `ChainedAssignmentError`. Создайте `subset = df[df["price"] > 0]`, измените `subset` — `df` не изменился. Примените `df.assign(revenue=pd.col("price") * pd.col("members"))` и `df.loc[pd.col("members") > 50]`. Ожидаемый вывод: `dtype` строки — `str`; `is_string_dtype` — `True`; `ChainedAssignmentError` при chained; `pd.col()` работает в `.assign()` и `.loc`. В комментариях объясните: почему `.append()` удалён в 2.0; как мигрировать через 2.3 (включить `pd.options.future.infer_string`); зачем `pd.options.future.infer_string` только в 2.3 (обратная совместимость); что произойдёт при `dtype == object` (не сравнится со `str`); как CoW влияет на производительность (меньше копирований); какие ещё breaking changes есть в 3.0 (`inplace` депрекация, `errors="ignore"` удалён).

**Задача 59 (мини-проект: EDA + очистка + FE).** Загрузите `clients.csv` (`parse_dates=["registered_at"]`), `memberships.csv`, `visits.csv` (`parse_dates=["visit_date"]`), `payments.csv`. Проведите EDA: `info()`, `describe()`, `value_counts()`, `isna().sum()`, `duplicated().sum()`. Очистка: `drop_duplicates(subset="client_id", keep="first")`, проверка диапазонов, ссылочной целостности через `.isin()`. Feature engineering: `merge` четырёх таблиц, `revenue = visits_count * price * (1 - discount)`, `profit = revenue - visits_count * cost`, `year/month/quarter` через `.dt`, `age_group` через `pd.cut`. Сохраните в `full.parquet`. Ожидаемый вывод: 51 пропуск в `email`, 10 дубликатов в `clients`; после очистки 1000 клиентов; итоговый DataFrame 50 000 × 15; Parquet сохраняет `category` и `datetime64[us]`. В комментариях объясните: зачем `parse_dates` при загрузке (тип `datetime` вместо строки); как проверить ссылочную целостность (`visits["client_id"].isin(clients["client_id"])`); почему `pd.cut` с фиксированными границами надёжнее `qcut` (не зависит от данных); как `merge` четырёх таблиц влияет на размер (декартово произведение при дубликатах); зачем сохранять в Parquet (типы, скорость); как `profit` считается с учётом скидки и себестоимости.

**Задача 60 (мини-проект: визуализация + выводы).** Используя `full.parquet` из задачи 59, ответьте на 5 бизнес-вопросов: динамика (`groupby(dt.to_period("M"))["revenue"].sum()` → `line`), залы (`groupby("hall", observed=True).agg(...)` → `barh`), топ-клиенты (`nlargest(10)`), статусы (`groupby("status").size()` → `pie`), города (`groupby("city")["revenue"].sum().sort_values()` → `bar`). Для каждого графика примените `figsize`, `title`, `grid`, `axhline` со средним; сохраните в PNG с `dpi=300`. Сформулируйте 3–5 бизнес-рекомендаций. Ожидаемый вывод: 5 графиков с осмысленными заголовками и линией среднего; 3–5 проверяемых рекомендаций с числами; указаны ограничения (корреляция ≠ причинность, сезонность, выбросы). В комментариях объясните: как аннотировать среднее на графике (`axhline`); почему `barh` удобнее для длинных названий залов и городов; как скользящее среднее помогает в динамике; какие ограничения у анализа (сезонность праздников, акции); как проверить отсутствие отрицательных `revenue`; как сравнить два года на одном графике через `pivot`.

---
